# 05 - Build supervised list

This notebook builds the final supervised gene list by intersecting first- and second-iteration unanimous predictions.

## Setup

In [1]:
from pathlib import Path

import pandas as pd


PROBABILITY_THRESHOLD = 0.9

PROJECT_ROOT = Path.cwd()

FIRST_ITERATION_PREDICTIONS_FILE = PROJECT_ROOT / "results" / "first_iteration" / "02.2_genes_pos_unanimity_with_proba.csv"
SECOND_ITERATION_PREDICTIONS_FILE = PROJECT_ROOT / "results" / "second_iteration" / "04.2_genes_pos_unanimity_with_proba_second_iteration.csv"

OUTPUT_DIR = PROJECT_ROOT / "results" / "supervised_list"
SUPERVISED_LIST_FILE = OUTPUT_DIR / "05.1_supervised_list.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load prediction lists

In [2]:
REQUIRED_COLUMNS = {"Gene", "Avg_Probability", "Unanimity_Prediction"}


def load_prediction_list(file_path: Path, label: str) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing {label} prediction file: {file_path}")

    predictions = pd.read_csv(file_path)
    missing_columns = REQUIRED_COLUMNS.difference(predictions.columns)
    if missing_columns:
        missing = ", ".join(sorted(missing_columns))
        raise ValueError(f"{label} prediction file is missing required columns: {missing}")

    predictions = predictions.copy()
    predictions["Gene"] = predictions["Gene"].astype(str)
    predictions["Avg_Probability"] = pd.to_numeric(predictions["Avg_Probability"], errors="raise")
    predictions["Unanimity_Prediction"] = predictions["Unanimity_Prediction"].astype(str).str.strip().str.lower()

    return predictions


first_iteration_predictions = load_prediction_list(FIRST_ITERATION_PREDICTIONS_FILE, "first iteration")
second_iteration_predictions = load_prediction_list(SECOND_ITERATION_PREDICTIONS_FILE, "second iteration")

print(f"First-iteration predictions: {len(first_iteration_predictions):,}")
print(f"Second-iteration predictions: {len(second_iteration_predictions):,}")

First-iteration predictions: 19,928
Second-iteration predictions: 19,928


## Build supervised list

In [3]:
UNANIMOUS_VALUES = {"1", "true", "yes", "si"}


first_iteration_unanimous = (
    first_iteration_predictions.loc[
        first_iteration_predictions["Unanimity_Prediction"].isin(UNANIMOUS_VALUES),
        ["Gene", "Avg_Probability"],
    ]
    .rename(columns={"Avg_Probability": "Avg_Probability_First_Iteration"})
    .drop_duplicates(subset="Gene")
)

second_iteration_unanimous = (
    second_iteration_predictions.loc[
        second_iteration_predictions["Unanimity_Prediction"].isin(UNANIMOUS_VALUES),
        ["Gene", "Avg_Probability"],
    ]
    .rename(columns={"Avg_Probability": "Avg_Probability_Second_Iteration"})
    .drop_duplicates(subset="Gene")
)

supervised_list = first_iteration_unanimous.merge(second_iteration_unanimous, on="Gene", how="inner")
supervised_list["Mean_Avg_Probability"] = supervised_list[
    ["Avg_Probability_First_Iteration", "Avg_Probability_Second_Iteration"]
].mean(axis=1)

supervised_list = (
    supervised_list.loc[supervised_list["Mean_Avg_Probability"] > PROBABILITY_THRESHOLD]
    .sort_values(["Mean_Avg_Probability", "Gene"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Genes with unanimous prediction in both iterations: {len(first_iteration_unanimous.merge(second_iteration_unanimous, on='Gene', how='inner')):,}")
print(f"Genes retained in supervised list: {len(supervised_list):,}")

supervised_list.head()

Genes with unanimous prediction in both iterations: 630
Genes retained in supervised list: 215


,Gene,Avg_Probability_First_Iteration,Avg_Probability_Second_Iteration,Mean_Avg_Probability
0,WBGene00010962,0.976902,0.974923,0.975912
1,WBGene00010964,0.977731,0.973328,0.975530
2,WBGene00000229,0.973648,0.971117,0.972383
3,WBGene00009161,0.972366,0.971074,0.971720
4,WBGene00009688,0.970132,0.971211,0.970672


## Save output

In [4]:
supervised_list.to_csv(SUPERVISED_LIST_FILE, index=False)

print(f"Saved supervised list to: {SUPERVISED_LIST_FILE}")

Saved supervised list to: C:\Users\masoz\Desktop\TesisMaestria\Papers\Paper1\scripts_publicacion\results\supervised_list\05.1_supervised_list.csv
